In [2]:
# Imports y configuración
import pandas as pd
import numpy as np
import json
import os
from pathlib import Path
from dotenv import load_dotenv
from sqlalchemy import create_engine

# Cargar credenciales desde .env local (en la misma carpeta que este notebook)
# override=True fuerza sobrescribir variables de entorno existentes
env_path = Path("/Users/cesaracosta/Documents/SPOT2/dagster/dagster-pipeline/src/dagster_pipeline/defs/conversation_analysis/notebooks/.env")

if env_path.exists():
    # override=True es crucial para sobrescribir valores None o vacíos del kernel
    load_dotenv(env_path, override=True)
    print(f"✅ Cargado .env desde: {env_path}")
else:
    print(f"❌ No existe: {env_path}")

# Configuración Geospot desde .env
GEOSPOT_HOST = os.getenv("SQL_SERVER_HOST_geo")
GEOSPOT_USER = os.getenv("SQL_SERVER_USER_geo")
GEOSPOT_PASSWORD = os.getenv("SQL_SERVER_PASSWORD_geo")
GEOSPOT_DB = os.getenv("SQL_SERVER_DB_geo")
GEOSPOT_PORT = os.getenv("SQL_SERVER_PORT_geo", "5432")
TABLE_NAME = "bt_conv_conversations"

print(f"✅ Imports cargados")
print(f"📊 Host: {GEOSPOT_HOST}")
print(f"📊 User: {GEOSPOT_USER}")
print(f"📊 Database: {GEOSPOT_DB}")
print(f"📊 Port: {GEOSPOT_PORT}")

✅ Cargado .env desde: /Users/cesaracosta/Documents/SPOT2/dagster/dagster-pipeline/src/dagster_pipeline/defs/conversation_analysis/notebooks/.env
✅ Imports cargados
📊 Host: gdtoxqaj1v5lvq.cqt3kzfdcy2o.us-east-1.rds.amazonaws.com
📊 User: cesar.acosta
📊 Database: geospot
📊 Port: 5432


In [3]:
# Conectar a Geospot y leer la tabla
engine_url = (
    f"postgresql+psycopg2://{GEOSPOT_USER}:{GEOSPOT_PASSWORD}"
    f"@{GEOSPOT_HOST}:{GEOSPOT_PORT}/{GEOSPOT_DB}"
)
engine = create_engine(engine_url)

print(f"📥 Leyendo tabla {TABLE_NAME}...")
with engine.connect() as conn:
    df = pd.read_sql(f"SELECT * FROM bt_conv_conversations", conn)

print(f"✅ Leídos {len(df)} registros")
print(f"📊 Columnas: {list(df.columns)}")
df.head()

📥 Leyendo tabla bt_conv_conversations...
✅ Leídos 378 registros
📊 Columnas: ['conv_id', 'lead_phone_number', 'conv_start_date', 'conv_end_date', 'conv_text', 'conv_messages', 'conv_last_message_date', 'lead_id', 'lead_created_at', 'conv_variables', 'vis_pseudo_id', 'spot_id', 'aud_inserted_date', 'aud_inserted_at', 'aud_updated_date', 'aud_updated_at', 'aud_job']


,conv_id,lead_phone_number,conv_start_date,conv_end_date,conv_text,conv_messages,conv_last_message_date,lead_id,lead_created_at,conv_variables,vis_pseudo_id,spot_id,aud_inserted_date,aud_inserted_at,aud_updated_date,aud_updated_at,aud_job
0,525541428397_20260108,525541428397,2026-01-08 09:16:51,2026-01-08 09:18:47,"[2026-01-08 09:16:51] user: ¡Hola, me interesa...","[{'id': 168153, 'type': 'user', 'message_body'...",2026-01-08 09:18:47,57154,2025-10-23 09:03:25,"[{'conv_variable_name': 'messages_count', 'con...",None,None,None,None,2026-01-13,2026-01-13 16:16:31.029115,conversation_analysis_pipeline
1,528441202017_20260109,528441202017,2026-01-09 09:56:51,2026-01-09 10:06:35,"[2026-01-09 09:56:51] user: ¡Hola, me interesa...","[{'id': 168934, 'type': 'user', 'message_body'...",2026-01-09 10:06:35,60918,2026-01-02 20:15:06,"[{'conv_variable_name': 'messages_count', 'con...",None,None,None,None,2026-01-13,2026-01-13 16:16:31.029115,conversation_analysis_pipeline
2,16154034371_20260107,16154034371,2026-01-07 14:34:35,2026-01-07 14:38:59,"[2026-01-07 14:34:35] user: ¡Hola, me interesa...","[{'id': 167598, 'type': 'user', 'message_body'...",2026-01-07 14:38:59,61168,2026-01-07 14:34:36,"[{'conv_variable_name': 'messages_count', 'con...",None,None,None,None,2026-01-13,2026-01-13 16:16:31.029115,conversation_analysis_pipeline
3,523310384825_20260108,523310384825,2026-01-08 16:33:22,2026-01-08 16:36:51,[2026-01-08 16:33:22] user: buenas tardes [202...,"[{'id': 168628, 'type': 'user', 'message_body'...",2026-01-08 16:36:51,61268,2026-01-08 16:32:55,"[{'conv_variable_name': 'messages_count', 'con...",None,None,None,None,2026-01-13,2026-01-13 16:16:31.029115,conversation_analysis_pipeline
4,522214293626_20260107,522214293626,2026-01-07 10:25:36,2026-01-07 10:28:23,"[2026-01-07 10:25:36] user: ¡Hola, me interesa...","[{'id': 167315, 'type': 'user', 'message_body'...",2026-01-07 10:28:23,61144,2026-01-07 10:25:37,"[{'conv_variable_name': 'messages_count', 'con...",None,None,None,None,2026-01-13,2026-01-13 16:16:31.029115,conversation_analysis_pipeline


In [4]:
# Extrae variable_name, variable_value y reasoning de la columna conv_variables
import ast

def extract_vars(conv_variables):
    # Algunos registros pueden tener valores None o no listas, los controlamos
    if not conv_variables:
        return []
    # conv_variables puede ser string (representando lista de dicts) o lista de dicts
    if isinstance(conv_variables, str):
        try:
            conv_variables = ast.literal_eval(conv_variables)
        except Exception:
            return []
    if not isinstance(conv_variables, list):
        return []
    # Extraemos los datos deseados
    result = []
    for var in conv_variables:
        result.append({
            'variable_category': var.get('conv_variable_category'),
            'variable_name': var.get('conv_variable_name'),
            'variable_value': var.get('conv_variable_value'),
            'reasoning': var.get('conv_variable_explanation')
        })
    return result

# Expandimos conv_variables por filas, asociando cada variable al conv_id y demás columnas
rows = []
for idx, row in df.iterrows():
    vars_data = extract_vars(row['conv_variables'])
    for var in vars_data:
        rows.append({
            'conv_id': row['conv_id'],
            'conv_start_date': row['conv_start_date'],
            'lead_phone_number': row['lead_phone_number'],
            'conv_text': row['conv_text'],
            'variable_category': var['variable_category'],
            'variable_name': var['variable_name'],
            'variable_value': var['variable_value'],
            'reasoning': var['reasoning']
        })

df_ai = pd.DataFrame(rows)
df_ai

,conv_id,conv_start_date,lead_phone_number,conv_text,variable_category,variable_name,variable_value,reasoning
0,525541428397_20260108,2026-01-08 09:16:51,525541428397,"[2026-01-08 09:16:51] user: ¡Hola, me interesa...",metric,messages_count,7.000000,None
1,525541428397_20260108,2026-01-08 09:16:51,525541428397,"[2026-01-08 09:16:51] user: ¡Hola, me interesa...",metric,user_messages_count,3.000000,None
2,525541428397_20260108,2026-01-08 09:16:51,525541428397,"[2026-01-08 09:16:51] user: ¡Hola, me interesa...",metric,total_time_conversation,0.032222,None
3,525541428397_20260108,2026-01-08 09:16:51,525541428397,"[2026-01-08 09:16:51] user: ¡Hola, me interesa...",metric,user_average_time_respond_minutes,0.483333,None
4,525541428397_20260108,2026-01-08 09:16:51,525541428397,"[2026-01-08 09:16:51] user: ¡Hola, me interesa...",tag,active_user,1.000000,None
...,...,...,...,...,...,...,...,...
4902,529842505302_20260111,2026-01-11 14:27:02,529842505302,[2026-01-11 14:27:02] user: Sí me interesa [20...,tag,engaged_user,0.000000,None
4903,529842505302_20260111,2026-01-11 14:27:02,529842505302,[2026-01-11 14:27:02] user: Sí me interesa [20...,tag,visit_intention,0.000000,None
4904,529842505302_20260111,2026-01-11 14:27:02,529842505302,[2026-01-11 14:27:02] user: Sí me interesa [20...,tag,scheduled_visit,0.000000,None
4905,529842505302_20260111,2026-01-11 14:27:02,529842505302,[2026-01-11 14:27:02] user: Sí me interesa [20...,tag,created_project,0.000000,None


In [5]:
df_ai[df_ai['lead_phone_number'] == '523325639479']

,conv_id,conv_start_date,lead_phone_number,conv_text,variable_category,variable_name,variable_value,reasoning
3557,523325639479_20260109,2026-01-09 17:01:56,523325639479,"[2026-01-09 17:01:56] user: ¡Hola, me interesa...",metric,messages_count,15.000000,None
3558,523325639479_20260109,2026-01-09 17:01:56,523325639479,"[2026-01-09 17:01:56] user: ¡Hola, me interesa...",metric,user_messages_count,6.000000,None
3559,523325639479_20260109,2026-01-09 17:01:56,523325639479,"[2026-01-09 17:01:56] user: ¡Hola, me interesa...",metric,total_time_conversation,0.197222,None
3560,523325639479_20260109,2026-01-09 17:01:56,523325639479,"[2026-01-09 17:01:56] user: ¡Hola, me interesa...",metric,user_average_time_respond_minutes,2.106667,None
3561,523325639479_20260109,2026-01-09 17:01:56,523325639479,"[2026-01-09 17:01:56] user: ¡Hola, me interesa...",tag,active_user,1.000000,None
3562,523325639479_20260109,2026-01-09 17:01:56,523325639479,"[2026-01-09 17:01:56] user: ¡Hola, me interesa...",tag,engaged_user,1.000000,None
3563,523325639479_20260109,2026-01-09 17:01:56,523325639479,"[2026-01-09 17:01:56] user: ¡Hola, me interesa...",tag,visit_intention,1.000000,None
3564,523325639479_20260109,2026-01-09 17:01:56,523325639479,"[2026-01-09 17:01:56] user: ¡Hola, me interesa...",tag,scheduled_visit,1.000000,None
3565,523325639479_20260109,2026-01-09 17:01:56,523325639479,"[2026-01-09 17:01:56] user: ¡Hola, me interesa...",tag,created_project,2.000000,None
3566,523325639479_20260109,2026-01-09 17:01:56,523325639479,"[2026-01-09 17:01:56] user: ¡Hola, me interesa...",ai_tag,missed_messages,1.000000,The assistant failed to understand the user's ...


In [6]:
df_ai.variable_name.value_counts()

variable_name
messages_count                       378
engaged_user                         378
user_messages_count                  378
scheduled_visit                      378
visit_intention                      378
created_project                      378
active_user                          378
user_average_time_respond_minutes    378
total_time_conversation              378
claim                                219
out_of_context                       219
churn                                219
scaling_agent                        219
assistant_error                      219
conversation_loop                    219
missed_messages                      159
is_broker                             32
Name: count, dtype: int64

In [4]:
df_ai.variable_name.value_counts()

variable_name
messages_count                       378
engaged_user                         378
user_messages_count                  378
scheduled_visit                      378
visit_intention                      378
created_project                      378
active_user                          378
user_average_time_respond_minutes    378
total_time_conversation              378
claim                                219
out_of_context                       219
churn                                219
scaling_agent                        219
assistant_error                      219
conversation_loop                    219
refinement                           159
is_broker                             32
Name: count, dtype: int64

In [7]:
df_ai[(df_ai['variable_name'] == 'missed_messages') & (df_ai['variable_value'] > 0)]

,conv_id,conv_start_date,lead_phone_number,conv_text,variable_category,variable_name,variable_value,reasoning
189,529992164795_20260110,2026-01-10 20:36:57,529992164795,"[2026-01-10 20:36:57] user: ¡Hola, me interesa...",ai_tag,missed_messages,1.0,The assistant failed to provide specific infor...
3356,15613761603_20260110,2026-01-10 22:59:38,15613761603,"[2026-01-10 22:59:38] user: ¡Hola, me interesa...",ai_tag,missed_messages,1.0,The assistant did not provide a relevant respo...
3406,522288556968_20260112,2026-01-12 12:53:30,522288556968,"[2026-01-12 12:53:30] user: ¡Hola, me interesa...",ai_tag,missed_messages,1.0,The assistant failed to provide additional det...
3446,523311404194_20260112,2026-01-12 09:06:07,523311404194,"[2026-01-12 09:06:07] user: ¡Hola, me interesa...",ai_tag,missed_messages,1.0,The assistant failed to provide specific infor...
3466,523314175489_20260110,2026-01-10 14:36:27,523314175489,"[2026-01-10 14:36:27] user: ¡Hola, me interesa...",ai_tag,missed_messages,1.0,The assistant failed to provide the exact loca...
3476,523317464247_20260109,2026-01-09 16:49:20,523317464247,"[2026-01-09 16:49:20] user: ¡Hola, me interesa...",ai_tag,missed_messages,1.0,The assistant could not provide the specific i...
3516,523319863447_20260112,2026-01-12 12:46:32,523319863447,"[2026-01-12 12:46:32] user: ¡Hola, me interesa...",ai_tag,missed_messages,2.0,The assistant repeated the same information in...
3546,523322541366_20260112,2026-01-12 13:51:33,523322541366,"[2026-01-12 13:51:33] user: ¡Hola, me interesa...",ai_tag,missed_messages,2.0,The assistant repeated itself and failed to pr...
3566,523325639479_20260109,2026-01-09 17:01:56,523325639479,"[2026-01-09 17:01:56] user: ¡Hola, me interesa...",ai_tag,missed_messages,1.0,The assistant failed to understand the user's ...
3576,523325639479_20260112,2026-01-12 11:23:37,523325639479,[2026-01-12 11:23:37] user: Agendar Cita Opció...,ai_tag,missed_messages,1.0,The assistant failed to acknowledge the user's...


In [16]:
df_ai[(df_ai['variable_category'] == 'ai_tag') & (df_ai['variable_value'] == 1) & (df_ai['lead_phone_number'] == '528120404063')]

,conv_id,conv_start_date,lead_phone_number,conv_text,variable_category,variable_name,variable_value,reasoning
5039,528120404063_20260111,2026-01-11 17:12:17,528120404063,"[2026-01-11 17:12:17] user: ¡Hola, me interesa...",ai_tag,conversation_loop,1.0,The assistant repeated the same question about...


In [10]:
df_ai.reasoning.value_counts()

reasoning
The assistant did not provide any error messages.                                                                          279
The assistant did not ask a specific question that went unanswered for over 24 hours.                                      248
The user did not express any complaints or frustrations.                                                                   245
The conversation is entirely related to real estate.                                                                       164
There was no indication that the request was escalated to a human agent.                                                   147
                                                                                                                          ... 
The user expresses dissatisfaction by stating 'no tienes pequeñas', indicating frustration with the options provided.        1
The conversation is focused on real estate, specifically on purchasing a small warehouse.            

In [17]:
df_ai

,conv_id,conv_start_date,lead_phone_number,conv_text,variable_category,variable_name,variable_value,reasoning
0,525541428397_20260108,2026-01-08 09:16:51,525541428397,"[2026-01-08 09:16:51] user: ¡Hola, me interesa...",metric,messages_count,7.000000,None
1,525541428397_20260108,2026-01-08 09:16:51,525541428397,"[2026-01-08 09:16:51] user: ¡Hola, me interesa...",metric,user_messages_count,3.000000,None
2,525541428397_20260108,2026-01-08 09:16:51,525541428397,"[2026-01-08 09:16:51] user: ¡Hola, me interesa...",metric,total_time_conversation,0.032222,None
3,525541428397_20260108,2026-01-08 09:16:51,525541428397,"[2026-01-08 09:16:51] user: ¡Hola, me interesa...",metric,user_average_time_respond_minutes,0.483333,None
4,525541428397_20260108,2026-01-08 09:16:51,525541428397,"[2026-01-08 09:16:51] user: ¡Hola, me interesa...",tag,active_user,1.000000,None
...,...,...,...,...,...,...,...,...
5185,573025470775_20260112,2026-01-12 17:02:07,573025470775,"[2026-01-12 17:02:07] user: ¡Hola, me interesa...",ai_tag,out_of_context,0.000000,"The conversation is related to real estate, sp..."
5186,573025470775_20260112,2026-01-12 17:02:07,573025470775,"[2026-01-12 17:02:07] user: ¡Hola, me interesa...",ai_tag,churn,0.000000,The assistant did not ask a specific question ...
5187,573025470775_20260112,2026-01-12 17:02:07,573025470775,"[2026-01-12 17:02:07] user: ¡Hola, me interesa...",ai_tag,scaling_agent,1.000000,The assistant mentioned that the request will ...
5188,573025470775_20260112,2026-01-12 17:02:07,573025470775,"[2026-01-12 17:02:07] user: ¡Hola, me interesa...",ai_tag,assistant_error,0.000000,The assistant did not provide any error messages.
